# Exp18 KC Learning Curves

This notebook builds per-KC timelines across problems, overlays V3 gap tags, fits constrained power-law curves, and compares those fits against a baseline that uses problem score alone.

The analysis starts with the three validated students and then expands to every available V3 learning-curve annotation file in `results/learning_curve/`.

Outputs are written to `results/human_validation/exp18_visualizations/`, `results/human_validation/exp18_learning_curve_summary.json`, and CSV files for downstream analysis.

In [26]:
import json
import os
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.optimize import curve_fit

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / 'dataset').exists() and (candidate / 'lib').exists():
        ROOT = candidate
        break
os.chdir(ROOT)

from utils.constants import MAINTABLE_PATH, PROBLEM_PROMPT_PATH

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 140

OUTPUT_DIR = ROOT / 'results' / 'human_validation'
FIG_DIR = OUTPUT_DIR / 'exp18_visualizations'
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TIMELINE_CSV = OUTPUT_DIR / 'exp18_kc_timeline_data.csv'
AGGREGATE_CSV = OUTPUT_DIR / 'exp18_kc_aggregate_curves.csv'
FIT_CSV = OUTPUT_DIR / 'exp18_kc_fit_metrics.csv'
SUMMARY_JSON = OUTPUT_DIR / 'exp18_learning_curve_summary.json'

FOCUS_STUDENTS = [10155, 14475, 14476]
MIN_PROBLEMS_PER_KC = 10
MIN_KC_PROBLEMS = 5

KC_ORDER = [
    'If/Else', 'NestedIf', 'While', 'For', 'NestedFor',
    'Math+-*/', 'Math%', 'LogicAndNotOr', 'LogicCompareNum', 'LogicBoolean',
    'StringFormat', 'StringConcat', 'StringIndex', 'StringLen',
    'StringEqual', 'CharEqual', 'ArrayIndex', 'DefFunction',
]


In [27]:
def is_required_kc(value):
    return pd.notna(value) and float(value) == 1.0

def get_best_attempts_with_timestamp(df):
    sort_cols = ['SubjectID', 'ProblemID', 'Score', 'ServerTimestamp', 'Attempt', 'Order']
    ranked = df.sort_values(
        sort_cols,
        ascending=[True, True, False, False, False, False],
        na_position='last',
    ).copy()
    return ranked.groupby(['SubjectID', 'ProblemID'], as_index=False).first()

main_cols = ['Order', 'SubjectID', 'ServerTimestamp', 'ProblemID', 'Attempt', 'CodeStateID', 'EventType', 'Score']
main_df = pd.read_csv(MAINTABLE_PATH, usecols=main_cols)
main_df = main_df[main_df['EventType'] == 'Run.Program'].copy()
main_df['ServerTimestamp'] = pd.to_datetime(main_df['ServerTimestamp'], errors='coerce', utc=True)
main_df['Score'] = pd.to_numeric(main_df['Score'], errors='coerce')
main_df['Attempt'] = pd.to_numeric(main_df['Attempt'], errors='coerce')
main_df['ProblemID'] = pd.to_numeric(main_df['ProblemID'], errors='coerce')
main_df['SubjectID'] = pd.to_numeric(main_df['SubjectID'], errors='coerce')
main_df['Order'] = pd.to_numeric(main_df['Order'], errors='coerce')
main_df = main_df.dropna(subset=['SubjectID', 'ProblemID', 'Score']).copy()
main_df['SubjectID'] = main_df['SubjectID'].astype(int)
main_df['ProblemID'] = main_df['ProblemID'].astype(int)

prompt_df = pd.read_csv(PROBLEM_PROMPT_PATH)
KC_COLUMNS = [kc for kc in KC_ORDER if kc in prompt_df.columns]
kc_problem_counts = {kc: int(prompt_df[kc].fillna(0).eq(1).sum()) for kc in KC_COLUMNS}
FIT_KC_COLUMNS = [kc for kc in KC_ORDER if kc in kc_problem_counts and kc_problem_counts[kc] >= MIN_PROBLEMS_PER_KC]
EXCLUDED_KC_COLUMNS = [kc for kc in KC_ORDER if kc in kc_problem_counts and kc_problem_counts[kc] < MIN_PROBLEMS_PER_KC]
kc_coverage_df = pd.DataFrame([
    {
        'KC': kc,
        'problem_count': kc_problem_counts[kc],
        'status': 'fit' if kc in FIT_KC_COLUMNS else 'excluded',
    }
    for kc in KC_COLUMNS
])
excluded_kc_summary_df = kc_coverage_df[kc_coverage_df['status'] == 'excluded'].sort_values(['problem_count', 'KC']).reset_index(drop=True)
problem_to_kcs = {}
kc_to_problems = {kc: [] for kc in KC_COLUMNS}
problem_meta = {}

for _, row in prompt_df.iterrows():
    pid = int(row['ProblemID'])
    required_kcs = [kc for kc in KC_COLUMNS if is_required_kc(row.get(kc))]
    problem_to_kcs[pid] = required_kcs
    problem_meta[pid] = {
        'assignment_id': int(row['AssignmentID']) if pd.notna(row.get('AssignmentID')) else None,
        'requirement': str(row.get('Requirement', '')),
    }
    for kc in required_kcs:
        kc_to_problems[kc].append(pid)

best_attempts_df = get_best_attempts_with_timestamp(main_df)
best_attempts_df['ServerTimestamp'] = pd.to_datetime(best_attempts_df['ServerTimestamp'], errors='coerce', utc=True)
best_attempts_df = best_attempts_df.sort_values(['SubjectID', 'ServerTimestamp', 'Attempt', 'Order', 'ProblemID']).reset_index(drop=True)

print(f'Run.Program rows: {len(main_df):,}')
print(f'Best attempts: {len(best_attempts_df):,}')
print(f'Students represented in best attempts: {best_attempts_df.SubjectID.nunique():,}')
print(f'Problems with KC tags: {len(problem_to_kcs):,}')
print(f'Fit-eligible KCs (>= {MIN_PROBLEMS_PER_KC} problems): {FIT_KC_COLUMNS}')
print(f'Excluded KCs (< {MIN_PROBLEMS_PER_KC} problems): {EXCLUDED_KC_COLUMNS}')
display(prompt_df.head())
display(kc_coverage_df.sort_values(['status', 'problem_count', 'KC'], ascending=[True, False, True]))
if not excluded_kc_summary_df.empty:
    display(excluded_kc_summary_df)


Run.Program rows: 69,627
Best attempts: 16,179
Students represented in best attempts: 413
Problems with KC tags: 50
Fit-eligible KCs (>= 10 problems): ['If/Else', 'NestedIf', 'For', 'Math+-*/', 'LogicAndNotOr', 'LogicCompareNum', 'StringFormat', 'StringIndex', 'ArrayIndex']
Excluded KCs (< 10 problems): ['While', 'NestedFor', 'Math%', 'LogicBoolean', 'StringConcat', 'StringLen', 'StringEqual', 'CharEqual', 'DefFunction']


,AssignmentID,ProblemID,Requirement,If/Else,NestedIf,While,For,NestedFor,Math+-*/,Math%,...,LogicCompareNum,LogicBoolean,StringFormat,StringConcat,StringIndex,StringLen,StringEqual,CharEqual,ArrayIndex,DefFunction
0,439,1,Write a function in Java that implements the f...,1.0,NaN,NaN,NaN,NaN,1.0,NaN,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,439,3,Write a function in Java that implements the f...,1.0,1.0,NaN,NaN,NaN,NaN,NaN,...,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,439,5,Write a function in Java that implements the f...,1.0,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,439,12,Write a function in Java that implements the f...,1.0,1.0,NaN,NaN,NaN,NaN,NaN,...,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,439,13,Write a function in Java that implements the f...,1.0,1.0,NaN,NaN,NaN,1.0,NaN,...,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,KC,problem_count,status
13,StringLen,9,excluded
14,StringEqual,7,excluded
9,LogicBoolean,6,excluded
6,Math%,5,excluded
11,StringConcat,5,excluded
15,CharEqual,4,excluded
4,NestedFor,4,excluded
2,While,3,excluded
17,DefFunction,2,excluded
0,If/Else,44,fit


,KC,problem_count,status
0,DefFunction,2,excluded
1,While,3,excluded
2,CharEqual,4,excluded
3,NestedFor,4,excluded
4,Math%,5,excluded
5,StringConcat,5,excluded
6,LogicBoolean,6,excluded
7,StringEqual,7,excluded
8,StringLen,9,excluded


In [28]:
def normalize_gap_list(value):
    if isinstance(value, list):
        return [str(x) for x in value if isinstance(x, str)]
    if isinstance(value, dict):
        gaps = value.get('gaps')
        if isinstance(gaps, list):
            return [str(x) for x in gaps if isinstance(x, str)]
        parsed = value.get('parsed_response')
        if isinstance(parsed, dict) and isinstance(parsed.get('knowledge_gaps'), list):
            return [str(x) for x in parsed['knowledge_gaps'] if isinstance(x, str)]
        if isinstance(value.get('knowledge_gaps'), list):
            return [str(x) for x in value['knowledge_gaps'] if isinstance(x, str)]
        raw = value.get('raw_response')
        if isinstance(raw, str) and raw.strip():
            try:
                parsed_raw = json.loads(raw)
                if isinstance(parsed_raw, dict) and isinstance(parsed_raw.get('knowledge_gaps'), list):
                    return [str(x) for x in parsed_raw['knowledge_gaps'] if isinstance(x, str)]
            except Exception:
                pass
    return []

def load_v3_annotation_file(path):
    with open(path, 'r') as f:
        payload = json.load(f)
    annotations = payload.get('annotations', payload)
    raw_responses = payload.get('raw_responses', {})
    gap_map = {}
    key_set = set(annotations.keys()) | set(raw_responses.keys())
    for pid in sorted(key_set, key=lambda x: int(x)):
        gaps = normalize_gap_list(annotations.get(pid, []))
        if not gaps: 
            gaps = normalize_gap_list(raw_responses.get(pid, []))
        gap_map[str(pid)] = sorted(set(gaps))
    return payload, gap_map

def annotation_path_for_student(student_id, prefer_validation=False):
    validation_path = ROOT / 'results' / 'human_validation' / f'llm_v3_annotations_{student_id}.json'
    lc_path = ROOT / 'results' / 'learning_curve' / f'llm_v3_lc_{student_id}.json'
    ordered = [validation_path, lc_path] if prefer_validation else [lc_path, validation_path]
    for path in ordered:
        if path.exists():
            return path
    return None

learning_curve_students = {int(p.stem.split('_')[-1]) for p in (ROOT / 'results' / 'learning_curve').glob('llm_v3_lc_*.json')}
available_annotation_students = sorted(learning_curve_students | set(FOCUS_STUDENTS))
annotation_lookup = {}
annotation_metadata = {}

for sid in available_annotation_students:
    path = annotation_path_for_student(sid, prefer_validation=sid in FOCUS_STUDENTS)
    if path is None:
        continue
    payload, gap_map = load_v3_annotation_file(path)
    annotation_lookup[sid] = gap_map
    annotation_metadata[sid] = {
        'source': path.name,
        'cluster': payload.get('cluster', 'Unknown'),
        'n_annotations': len(gap_map),
    }

print(f'Annotation students found: {len(annotation_lookup)}')
display(pd.DataFrame([{'student_id': sid, **meta} for sid, meta in annotation_metadata.items()]).sort_values('student_id').head(10))


Annotation students found: 28


,student_id,source,cluster,n_annotations
0,106,llm_v3_lc_106.json,Average,20
1,9948,llm_v3_lc_9948.json,Struggling,39
2,10083,llm_v3_lc_10083.json,Average,27
3,10155,llm_v3_annotations_10155.json,Unknown,46
4,10224,llm_v3_lc_10224.json,Average,25
5,13365,llm_v3_lc_13365.json,High Performer,42
6,14186,llm_v3_lc_14186.json,Average,13
7,14189,llm_v3_lc_14189.json,Struggling,48
8,14205,llm_v3_lc_14205.json,High Performer,50
9,14289,llm_v3_lc_14289.json,High Performer,31


In [29]:
fit_kc_columns = globals().get('FIT_KC_COLUMNS')
if fit_kc_columns is None:
    fit_kc_columns = [
        kc for kc in KC_COLUMNS
        if kc in prompt_df.columns and int(prompt_df[kc].fillna(0).eq(1).sum()) >= MIN_PROBLEMS_PER_KC
    ]

def build_kc_timelines(best_attempts, annotations, kc_columns):
    rows = []
    for student_id, student_df in best_attempts.groupby('SubjectID'):
        student_df = student_df.sort_values(['ServerTimestamp', 'Attempt', 'Order', 'ProblemID']).copy()
        student_ann = annotations.get(int(student_id), {})
        if not student_ann:
            continue
        for kc in kc_columns:
            required_problems = set(kc_to_problems.get(kc, []))
            if not required_problems:
                continue
            kc_df = student_df[student_df['ProblemID'].isin(required_problems)].copy()
            if len(kc_df) < MIN_KC_PROBLEMS:
                continue
            kc_df = kc_df.sort_values(['ServerTimestamp', 'Attempt', 'Order', 'ProblemID']).reset_index(drop=True)
            for attempt_number, (_, row) in enumerate(kc_df.iterrows(), start=1):
                problem_id = int(row['ProblemID'])
                score = float(row['Score'])
                gaps = student_ann.get(str(problem_id), [])
                v3_flagged_gap = kc in gaps
                meta = problem_meta.get(problem_id, {})
                rows.append({
                    'SubjectID': int(student_id),
                    'KC': kc,
                    'attempt_number': attempt_number,
                    'ProblemID': problem_id,
                    'ServerTimestamp': row['ServerTimestamp'],
                    'Score': score,
                    'baseline_error': float(np.clip(1.0 - score, 0.0, 1.0)),
                    'v3_error': 1.0 if v3_flagged_gap else 0.0,
                    'V3FlaggedGap': bool(v3_flagged_gap),
                    'assignment_id': meta.get('assignment_id'),
                    'requirement': meta.get('requirement', ''),
                    'problem_kc_count': len(problem_to_kcs.get(problem_id, [])),
                })
    return pd.DataFrame(rows)

timeline_df = build_kc_timelines(best_attempts_df, annotation_lookup, fit_kc_columns)
timeline_df = timeline_df.sort_values(['SubjectID', 'KC', 'attempt_number']).reset_index(drop=True)
timeline_df['gap_count'] = timeline_df.groupby(['SubjectID', 'KC'])['V3FlaggedGap'].transform('sum')
timeline_df['score_std'] = timeline_df.groupby(['SubjectID', 'KC'])['Score'].transform('std').fillna(0.0)
timeline_df['score_error'] = timeline_df['baseline_error']

print(f'Timeline rows: {len(timeline_df):,}')
print(f'Students represented in timelines: {timeline_df.SubjectID.nunique():,}')
print(f'KCs represented: {timeline_df.KC.nunique():,}')
display(timeline_df.head(10))


Timeline rows: 4,377
Students represented in timelines: 28
KCs represented: 9


,SubjectID,KC,attempt_number,ProblemID,ServerTimestamp,Score,baseline_error,v3_error,V3FlaggedGap,assignment_id,requirement,problem_kc_count,gap_count,score_std,score_error
0,106,If/Else,1,13,2019-02-24 23:23:05+00:00,1.0,0.0,0.0,False,439,Write a function in Java that implements the f...,6,0,0.042592,0.0
1,106,If/Else,2,232,2019-02-24 23:29:27+00:00,1.0,0.0,0.0,False,439,"Given a day of the week encoded as 0 = Sun, 1 ...",6,0,0.042592,0.0
2,106,If/Else,3,235,2019-02-24 23:34:00+00:00,1.0,0.0,0.0,False,439,You and your date are trying to get a table at...,3,0,0.042592,0.0
3,106,If/Else,4,234,2019-02-24 23:36:34+00:00,1.0,0.0,0.0,False,439,"When squirrels get together for a party, they ...",5,0,0.042592,0.0
4,106,If/Else,5,236,2019-02-24 23:40:53+00:00,1.0,0.0,0.0,False,439,"You have a green lottery ticket, with ints a, ...",3,0,0.042592,0.0
5,106,If/Else,6,5,2019-02-24 23:44:51+00:00,1.0,0.0,0.0,False,439,Write a function in Java that implements the f...,3,0,0.042592,0.0
6,106,If/Else,7,233,2019-02-24 23:47:53+00:00,1.0,0.0,0.0,False,439,The number 6 is a truly great number. Given tw...,4,0,0.042592,0.0
7,106,If/Else,8,1,2019-02-24 23:52:21+00:00,1.0,0.0,0.0,False,439,Write a function in Java that implements the f...,4,0,0.042592,0.0
8,106,If/Else,9,3,2019-02-24 23:59:04+00:00,1.0,0.0,0.0,False,439,Write a function in Java that implements the f...,5,0,0.042592,0.0
9,106,If/Else,10,12,2019-02-25 00:03:06+00:00,1.0,0.0,0.0,False,439,Write a function in Java that implements the f...,5,0,0.042592,0.0


In [30]:
def power_law(x, a, b):
    return a * np.power(x, b)

def fit_power_law(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = np.clip(y[mask], 0.0, 1.0)
    if len(x) < 3:
        return None
    if np.allclose(y, y[0]):
        a = float(np.clip(y[0], 0.0, 1.0))
        b = 0.0
        yhat = np.full_like(y, a, dtype=float)
    else:
        p0 = [float(np.clip(y[0], 0.05, 1.0)), -0.5]
        bounds = ([0.0, -5.0], [1.0, 0.0])
        try:
            popt, _ = curve_fit(power_law, x, y, p0=p0, bounds=bounds, maxfev=20000)
            a, b = float(popt[0]), float(popt[1])
            yhat = power_law(x, a, b)
        except Exception:
            a = float(np.clip(y.mean(), 0.0, 1.0))
            b = 0.0
            yhat = np.full_like(y, a, dtype=float)
    rmse = float(np.sqrt(np.mean((y - yhat) ** 2)))
    ss_res = float(np.sum((y - yhat) ** 2))
    ss_tot = float(np.sum((y - y.mean()) ** 2))
    if ss_tot == 0.0:
        r2 = 1.0 if ss_res == 0.0 else 0.0
    else:
        r2 = float(1.0 - ss_res / ss_tot)
    return {'a': a, 'b': b, 'rmse': rmse, 'r2': r2, 'n_points': int(len(x)), 'pred_error': yhat}

fit_rows = []
for (student_id, kc), group in timeline_df.groupby(['SubjectID', 'KC']):
    group = group.sort_values('attempt_number')
    for method, error_col in [('baseline', 'baseline_error'), ('v3', 'v3_error')]:
        fit = fit_power_law(group['attempt_number'].to_numpy(), group[error_col].to_numpy())
        if fit is None:
            continue
        fit_rows.append({
            'SubjectID': int(student_id),
            'KC': kc,
            'method': method,
            'n_points': fit['n_points'],
            'a': fit['a'],
            'b': fit['b'],
            'rmse': fit['rmse'],
            'r2': fit['r2'],
            'mean_score': float(group['Score'].mean()),
            'mean_baseline_error': float(group['baseline_error'].mean()),
            'mean_v3_error': float(group['v3_error'].mean()),
            'gap_rate': float(group['V3FlaggedGap'].mean()),
        })

fit_df = pd.DataFrame(fit_rows)

aggregate_rows = []
for method, error_col in [('baseline', 'baseline_error'), ('v3', 'v3_error')]:
    agg = (
        timeline_df
        .groupby(['KC', 'attempt_number'], as_index=False)
        .agg(
            mean_error=(error_col, 'mean'),
            std_error=(error_col, 'std'),
            n_students=('SubjectID', 'nunique'),
        )
    )
    agg['method'] = method
    aggregate_rows.append(agg)

aggregate_df = pd.concat(aggregate_rows, ignore_index=True)

aggregate_fit_rows = []
for (kc, method), group in aggregate_df.groupby(['KC', 'method']):
    fit = fit_power_law(group['attempt_number'].to_numpy(), group['mean_error'].to_numpy())
    if fit is None:
        continue
    aggregate_fit_rows.append({
        'KC': kc,
        'method': method,
        'n_points': fit['n_points'],
        'a': fit['a'],
        'b': fit['b'],
        'rmse': fit['rmse'],
        'r2': fit['r2'],
    })

aggregate_fit_df = pd.DataFrame(aggregate_fit_rows)

print(f'Per-timeline fits: {len(fit_df):,}')
print(f'Aggregate curve rows: {len(aggregate_df):,}')
print(f'Aggregate fits: {len(aggregate_fit_df):,}')
display(fit_df.head(10))


Per-timeline fits: 442
Aggregate curve rows: 432
Aggregate fits: 18


,SubjectID,KC,method,n_points,a,b,rmse,r2,mean_score,mean_baseline_error,mean_v3_error,gap_rate
0,106,If/Else,baseline,20,0.009524,-3.531763e-12,0.041513,-2.202682e-13,0.990476,0.009524,0.000000,0.000000
1,106,If/Else,v3,20,0.000000,0.000000e+00,0.000000,1.000000e+00,0.990476,0.009524,0.000000,0.000000
2,106,LogicAndNotOr,baseline,16,0.011905,-4.438411e-13,0.046107,-3.375078e-14,0.988095,0.011905,0.062500,0.062500
3,106,LogicAndNotOr,v3,16,0.062500,-4.115745e-13,0.242061,-3.130829e-14,0.988095,0.011905,0.062500,0.062500
4,106,LogicCompareNum,baseline,19,0.010025,-2.492402e-12,0.042533,-1.572076e-13,0.989975,0.010025,0.052632,0.052632
5,106,LogicCompareNum,v3,19,0.052632,-1.347341e-12,0.223297,-8.526513e-14,0.989975,0.010025,0.052632,0.052632
6,106,Math+-*/,baseline,10,0.019048,-3.159351e-18,0.057143,0.000000e+00,0.980952,0.019048,0.100000,0.100000
7,106,Math+-*/,v3,10,0.100000,-4.233208e-14,0.300000,-3.996803e-15,0.980952,0.019048,0.100000,0.100000
8,106,NestedIf,baseline,10,0.000000,0.000000e+00,0.000000,1.000000e+00,1.000000,0.000000,0.000000,0.000000
9,106,NestedIf,v3,10,0.000000,0.000000e+00,0.000000,1.000000e+00,1.000000,0.000000,0.000000,0.000000


## Section: Binned Gap Rate Analysis

For each fit-eligible student-KC timeline, split the problems into three equal chronological bins: early, mid, and late. Then compute per-bin V3 gap rates and a learning signal defined as early gap rate minus late gap rate.

In [ ]:
gap_rate_summary_json = OUTPUT_DIR / 'exp18_gap_rate_summary.json'

fit_kc_columns_local = globals().get('fit_kc_columns')
if fit_kc_columns_local is None:
    fit_kc_columns_local = globals().get('FIT_KC_COLUMNS')
if fit_kc_columns_local is None:
    fit_kc_columns_local = [
        kc for kc in KC_COLUMNS
        if kc in prompt_df.columns and int(prompt_df[kc].fillna(0).eq(1).sum()) >= MIN_PROBLEMS_PER_KC
    ]

def split_into_time_bins(group):
    ordered = group.sort_values('attempt_number').reset_index(drop=True).copy()
    if len(ordered) < 3:
        return []
    bin_indices = np.array_split(np.arange(len(ordered)), 3)
    bin_labels = ['early', 'mid', 'late']
    rows = []
    for bin_label, indices in zip(bin_labels, bin_indices):
        subset = ordered.iloc[indices]
        gap_rate = float(subset['V3FlaggedGap'].mean()) if len(subset) else np.nan
        rows.append({
            'bin': bin_label,
            'gap_rate': gap_rate,
            'n_bin_problems': int(len(subset)),
        })
    rows[0]['learning_signal'] = rows[0]['gap_rate'] - rows[2]['gap_rate']
    return rows

gap_rate_rows = []
student_heatmap_records = {}

for (student_id, kc), group in timeline_df.groupby(['SubjectID', 'KC']):
    if kc not in fit_kc_columns_local:
        continue
    binned_rows = split_into_time_bins(group)
    if not binned_rows:
        continue
    bin_map = {row['bin']: row for row in binned_rows}
    early_gap_rate = bin_map['early']['gap_rate']
    mid_gap_rate = bin_map['mid']['gap_rate']
    late_gap_rate = bin_map['late']['gap_rate']
    learning_signal = early_gap_rate - late_gap_rate
    pair_row = {
        'SubjectID': int(student_id),
        'KC': kc,
        'n_problems': int(len(group)),
        'early_gap_rate': early_gap_rate,
        'mid_gap_rate': mid_gap_rate,
        'late_gap_rate': late_gap_rate,
        'learning_signal': learning_signal,
    }
    gap_rate_rows.append(pair_row)
    student_heatmap_records.setdefault(int(student_id), {})[kc] = {
        'early': early_gap_rate,
        'mid': mid_gap_rate,
        'late': late_gap_rate,
        'learning_signal': learning_signal,
        'n_problems': int(len(group)),
    }

gap_rate_pair_df = pd.DataFrame(gap_rate_rows)
if gap_rate_pair_df.empty:
    raise ValueError('No fit-eligible student-KC pairs available for binned gap-rate analysis.')

kc_gap_summary_df = (
    gap_rate_pair_df.groupby('KC', as_index=False)
    .agg(
        mean_learning_signal=('learning_signal', 'mean'),
        median_learning_signal=('learning_signal', 'median'),
        student_count=('SubjectID', 'nunique'),
        positive_count=('learning_signal', lambda s: int((s > 1e-12).sum())),
        zero_count=('learning_signal', lambda s: int((np.abs(s) <= 1e-12).sum())),
        negative_count=('learning_signal', lambda s: int((s < -1e-12).sum())),
        mean_early_gap_rate=('early_gap_rate', 'mean'),
        mean_mid_gap_rate=('mid_gap_rate', 'mean'),
        mean_late_gap_rate=('late_gap_rate', 'mean'),
        mean_n_problems=('n_problems', 'mean'),
    )
    .sort_values('mean_learning_signal', ascending=False)
    .reset_index(drop=True)
)

kc_visual_order = kc_gap_summary_df['KC'].tolist()
bin_cols = ['early_gap_rate', 'mid_gap_rate', 'late_gap_rate']
bin_labels = ['early', 'mid', 'late']
mean_gap_matrix = kc_gap_summary_df.set_index('KC').loc[kc_visual_order, bin_cols]
student_signal_matrix = (
    gap_rate_pair_df.pivot(index='KC', columns='SubjectID', values='learning_signal')
    .reindex(kc_visual_order)
)
student_order = (
    gap_rate_pair_df.groupby('SubjectID')['learning_signal']
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)
student_signal_matrix = student_signal_matrix.reindex(columns=student_order)

signal_abs_max = float(np.nanmax(np.abs(student_signal_matrix.to_numpy())))
signal_abs_max = max(0.25, min(0.75, signal_abs_max))
summary_abs_max = max(0.10, float(np.nanmax(np.abs(kc_gap_summary_df['mean_learning_signal']))))
max_gap_rate = max(0.25, float(np.nanmax(mean_gap_matrix.to_numpy())))

from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D

gap_rate_cmap = LinearSegmentedColormap.from_list('gap_rate', ['#fbf7e8', '#b9dfc1', '#3aa6a8', '#1f4e79'])
signal_cmap = LinearSegmentedColormap.from_list('gap_signal', ['#b75d3a', '#f4efe3', '#2a7f62'])

plt.rcParams.update({
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
})

fig, axes = plt.subplots(
    1, 3,
    figsize=(16, max(6.2, 0.48 * len(kc_visual_order) + 2.2)),
    gridspec_kw={'width_ratios': [1.0, 1.05, 1.15]},
)

ax = axes[0]
im = ax.imshow(mean_gap_matrix.to_numpy(), aspect='auto', cmap=gap_rate_cmap, vmin=0, vmax=max_gap_rate)
ax.set_xticks(np.arange(len(bin_labels)))
ax.set_xticklabels(bin_labels)
ax.set_yticks(np.arange(len(kc_visual_order)))
ax.set_yticklabels(kc_visual_order)
ax.set_title('Mean V3 gap rate by phase')
for i, kc in enumerate(kc_visual_order):
    for j, col in enumerate(bin_cols):
        value = mean_gap_matrix.loc[kc, col]
        color = 'white' if value > max_gap_rate * 0.62 else '#263238'
        ax.text(j, i, f'{value:.0%}', ha='center', va='center', color=color, fontsize=8)
ax.set_xlabel('Chronological third of KC exposure')
ax.set_ylabel('Knowledge Component')
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
cbar.set_label('Gap rate')

ax = axes[1]
y = np.arange(len(kc_visual_order))
signals = kc_gap_summary_df.set_index('KC').loc[kc_visual_order, 'mean_learning_signal']
colors = np.where(signals >= 0, '#2a7f62', '#b75d3a')
ax.barh(y, signals, color=colors, alpha=0.9)
ax.axvline(0, color='#333333', lw=1)
ax.set_yticks(y)
ax.set_yticklabels([])
ax.invert_yaxis()
ax.set_xlim(-summary_abs_max * 1.25, summary_abs_max * 1.25)
ax.set_title('Learning signal by KC')
ax.set_xlabel('Early gap rate minus late gap rate')
for idx, value in enumerate(signals):
    ha = 'left' if value >= 0 else 'right'
    offset = 0.006 if value >= 0 else -0.006
    ax.text(value + offset, idx, f'{value:+.0%}', va='center', ha=ha, fontsize=8)
ax.grid(axis='x', alpha=0.22)
ax.grid(axis='y', visible=False)

ax = axes[2]
stack = kc_gap_summary_df.set_index('KC').loc[kc_visual_order, ['positive_count', 'zero_count', 'negative_count']]
left = np.zeros(len(stack))
stack_colors = {'positive_count': '#2a7f62', 'zero_count': '#b8b8b8', 'negative_count': '#b75d3a'}
stack_labels = {'positive_count': 'declined', 'zero_count': 'unchanged', 'negative_count': 'emerged/persisted'}
for col in ['positive_count', 'zero_count', 'negative_count']:
    ax.barh(y, stack[col], left=left, color=stack_colors[col], label=stack_labels[col], alpha=0.9)
    left += stack[col].to_numpy()
ax.set_yticks(y)
ax.set_yticklabels([])
ax.invert_yaxis()
ax.set_title('Student-KC direction counts')
ax.set_xlabel('Number of students')
ax.legend(loc='lower right', frameon=True)
ax.grid(axis='x', alpha=0.22)
ax.grid(axis='y', visible=False)

fig.suptitle('Exp18: LLM-flagged Java CS1 knowledge gaps across repeated KC exposure', y=1.02, fontsize=15)
fig.text(
    0.5, 0.985,
    'Positive signal means V3 flagged the KC more often early than late; negative signal means gaps persisted or appeared later.',
    ha='center', va='top', fontsize=10, color='#444444'
)
fig.tight_layout(rect=(0, 0, 1, 0.94))
gap_overview_path = FIG_DIR / 'exp18_gap_trajectory_overview.png'
fig.savefig(gap_overview_path, bbox_inches='tight')
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(15.5, max(5.8, 0.42 * len(kc_visual_order) + 1.8)))
im = ax.imshow(student_signal_matrix.to_numpy(), aspect='auto', cmap=signal_cmap, vmin=-signal_abs_max, vmax=signal_abs_max)
ax.set_xticks(np.arange(len(student_order)))
ax.set_xticklabels([str(sid) for sid in student_order], rotation=90)
ax.set_yticks(np.arange(len(kc_visual_order)))
ax.set_yticklabels(kc_visual_order)
ax.set_title('Student-by-KC learning signal matrix')
ax.set_xlabel('Student sorted by average learning signal')
ax.set_ylabel('Knowledge Component')
for label in ax.get_xticklabels():
    if int(label.get_text()) in FOCUS_STUDENTS:
        label.set_fontweight('bold')
        label.set_color('#1f4e79')
for sid in FOCUS_STUDENTS:
    if sid in student_order:
        xpos = student_order.index(sid)
        ax.axvline(xpos, color='#1f4e79', lw=0.9, alpha=0.7)
cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label('Early gap rate - late gap rate')
legend_handles = [Line2D([0], [0], color='#1f4e79', lw=1.5, label='validated student')]
ax.legend(handles=legend_handles, loc='upper right', frameon=True)
fig.tight_layout()
signal_matrix_path = FIG_DIR / 'exp18_student_gap_signal_matrix.png'
fig.savefig(signal_matrix_path, bbox_inches='tight')
plt.show()
plt.close(fig)

visualization_files = [str(gap_overview_path), str(signal_matrix_path)]

gap_rate_summary_payload = {
    'timelines': {
        'rows': int(len(gap_rate_pair_df)),
        'student_kc_pairs': int(gap_rate_pair_df[['SubjectID', 'KC']].drop_duplicates().shape[0]),
        'fit_kcs': fit_kc_columns_local,
    },
    'paths': {
        'summary_json': str(gap_rate_summary_json),
        'figure_dir': str(FIG_DIR),
    },
    'student_kc_pairs': gap_rate_pair_df.sort_values(['SubjectID', 'KC']).to_dict(orient='records'),
    'kc_summary': kc_gap_summary_df.to_dict(orient='records'),
    'visualization_files': visualization_files,
}

with open(gap_rate_summary_json, 'w') as f:
    json.dump(gap_rate_summary_payload, f, indent=2, default=lambda obj: obj.item() if hasattr(obj, 'item') else str(obj))

print(f'Saved {gap_rate_summary_json}')
print(f'Saved overview figures to {FIG_DIR}')
display(kc_gap_summary_df)
display(gap_rate_pair_df.head(12))


In [ ]:
student_kc_stats = (
    timeline_df.groupby(['SubjectID', 'KC'], as_index=False)
    .agg(
        n_points=('attempt_number', 'count'),
        gap_count=('V3FlaggedGap', 'sum'),
        score_std=('Score', 'std'),
        mean_score=('Score', 'mean'),
    )
)
student_kc_stats['score_std'] = student_kc_stats['score_std'].fillna(0.0)

def select_example_kc(student_id):
    subset = student_kc_stats[student_kc_stats['SubjectID'] == student_id].copy()
    subset = subset[subset['n_points'] >= MIN_KC_PROBLEMS]
    if subset.empty:
        return None
    subset['priority'] = subset['gap_count'] * 4.0 + subset['score_std'] * 6.0 + np.log1p(subset['n_points'])
    return subset.sort_values(['priority', 'gap_count', 'score_std', 'n_points'], ascending=[False, False, False, False]).iloc[0]

def phase_spans(group):
    ordered = group.sort_values('attempt_number').reset_index(drop=True)
    spans = []
    for label, indices in zip(['early', 'mid', 'late'], np.array_split(np.arange(len(ordered)), 3)):
        if len(indices) == 0:
            continue
        subset = ordered.iloc[indices]
        spans.append((label, int(subset['attempt_number'].min()), int(subset['attempt_number'].max()), float(subset['V3FlaggedGap'].mean())))
    return spans

def plot_example_student(ax, student_id, kc):
    group = timeline_df[(timeline_df['SubjectID'] == student_id) & (timeline_df['KC'] == kc)].sort_values('attempt_number').copy()
    x = group['attempt_number'].to_numpy()
    gap_mask = group['V3FlaggedGap'].to_numpy(dtype=bool)
    spans = phase_spans(group)
    phase_colors = {'early': '#f7ecd2', 'mid': '#e8f0e3', 'late': '#e3eef5'}
    for label, xmin, xmax, rate in spans:
        ax.axvspan(xmin - 0.5, xmax + 0.5, color=phase_colors[label], alpha=0.65, lw=0)
        ax.text((xmin + xmax) / 2, 0.04, f'{label}\n{rate:.0%}', ha='center', va='bottom', fontsize=8, color='#4a4a4a')
    ax.plot(x, group['Score'], color='#4d4d4d', marker='o', lw=1.8, ms=4.5, label='observed score')
    if gap_mask.any():
        ax.scatter(x[gap_mask], group.loc[gap_mask, 'Score'], color='#c93f32', edgecolor='white', linewidth=0.6, s=80, zorder=5, label='V3 flagged gap')
        ax.vlines(x[gap_mask], 0, group.loc[gap_mask, 'Score'], color='#c93f32', alpha=0.23, lw=1.4)
    rolling = group['Score'].rolling(window=min(5, len(group)), min_periods=1, center=True).mean()
    ax.plot(x, rolling, color='#1f4e79', lw=2.3, label='smoothed score')
    ax.set_ylim(-0.03, 1.05)
    ax.set_xlim(0.5, x.max() + 0.5)
    ax.set_xlabel('KC occurrence number')
    ax.set_ylabel('Score')
    early = spans[0][3]
    late = spans[-1][3]
    signal = early - late
    ax.set_title(f'Student {student_id} | {kc} | {len(group)} problems | signal {signal:+.0%}', fontsize=11)
    ax.grid(axis='y', alpha=0.25)
    ax.grid(axis='x', alpha=0.10)

example_specs = []
for sid in FOCUS_STUDENTS:
    spec = select_example_kc(sid)
    if spec is not None:
        example_specs.append(spec)

if example_specs:
    example_specs_df = pd.DataFrame(example_specs).reset_index(drop=True)
    display(example_specs_df[['SubjectID', 'KC', 'n_points', 'gap_count', 'score_std', 'mean_score']])
    fig, axes = plt.subplots(len(example_specs_df), 1, figsize=(13, 3.8 * len(example_specs_df)), sharex=False)
    if len(example_specs_df) == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, example_specs_df.iterrows()):
        plot_example_student(ax, int(row['SubjectID']), row['KC'])
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.01))
    fig.suptitle('Validated-student KC timelines with V3 gap markers and early/mid/late gap rates', y=1.035, fontsize=14)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    example_path = FIG_DIR / 'exp18_validated_student_gap_timelines.png'
    fig.savefig(example_path, bbox_inches='tight')
    plt.show()
    print(f'Saved {example_path}')
else:
    print('No example KC timelines met the minimum data threshold.')


In [ ]:
def plot_aggregate_kc(ax, kc):
    subset = aggregate_df[aggregate_df['KC'] == kc].copy()
    if subset.empty:
        ax.axis('off')
        return
    method_specs = {
        'baseline': {'label': 'score error baseline', 'color': '#555555', 'marker': 'o', 'linestyle': '-'},
        'v3': {'label': 'V3 gap rate', 'color': '#1f77b4', 'marker': 's', 'linestyle': '--'},
    }
    for method, spec in method_specs.items():
        group = subset[subset['method'] == method].sort_values('attempt_number')
        if group.empty:
            continue
        ax.plot(
            group['attempt_number'], group['mean_error'],
            marker=spec['marker'], color=spec['color'], lw=1.4, ms=3.2, alpha=0.72,
            label=spec['label']
        )
        fit_row = aggregate_fit_df[(aggregate_fit_df['KC'] == kc) & (aggregate_fit_df['method'] == method)]
        if not fit_row.empty:
            fit_row = fit_row.iloc[0]
            xfit = np.linspace(1, group['attempt_number'].max(), 200)
            ax.plot(xfit, power_law(xfit, fit_row['a'], fit_row['b']), linestyle=spec['linestyle'], color=spec['color'], lw=2.0)
    n_students = int(timeline_df[timeline_df['KC'] == kc]['SubjectID'].nunique())
    ax.set_title(f'{kc} (n={n_students})', fontsize=10)
    ax.set_ylim(0, min(0.55, max(0.18, subset['mean_error'].max() * 1.25)))
    ax.set_xlim(left=1)
    ax.grid(True, alpha=0.22)

fit_kc_columns = globals().get('FIT_KC_COLUMNS')
if fit_kc_columns is None:
    fit_kc_columns = [
        kc for kc in KC_COLUMNS
        if kc in prompt_df.columns and int(prompt_df[kc].fillna(0).eq(1).sum()) >= MIN_PROBLEMS_PER_KC
    ]

excluded_kc_columns = globals().get('EXCLUDED_KC_COLUMNS')
if excluded_kc_columns is None:
    excluded_kc_columns = [kc for kc in KC_COLUMNS if kc not in fit_kc_columns]

kc_problem_counts_local = globals().get('kc_problem_counts')
if kc_problem_counts_local is None:
    kc_problem_counts_local = {kc: int(prompt_df[kc].fillna(0).eq(1).sum()) for kc in KC_COLUMNS}

excluded_kc_summary_df = globals().get('excluded_kc_summary_df')
if excluded_kc_summary_df is None:
    excluded_kc_summary_df = pd.DataFrame([
        {
            'KC': kc,
            'problem_count': kc_problem_counts_local.get(kc, 0),
            'status': 'excluded',
        }
        for kc in excluded_kc_columns
    ]).sort_values(['problem_count', 'KC']).reset_index(drop=True)

n_rows = int(np.ceil(len(fit_kc_columns) / 3))
fig, axes = plt.subplots(n_rows, 3, figsize=(15.5, max(3.4 * n_rows, 6)), sharex=False, sharey=False)
axes = axes.flatten()
for ax, kc in zip(axes, fit_kc_columns):
    plot_aggregate_kc(ax, kc)
for ax in axes[len(fit_kc_columns):]:
    ax.axis('off')
for ax in axes[::3]:
    ax.set_ylabel('Mean rate')
for ax in axes[-3:]:
    ax.set_xlabel('KC occurrence number')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.015))
fig.suptitle('Aggregate KC trajectories: score error baseline compared with V3 gap rate', y=1.035, fontsize=14)
fig.tight_layout(rect=(0, 0, 1, 0.96))
aggregate_path = FIG_DIR / 'exp18_aggregate_gap_vs_score_curves.png'
fig.savefig(aggregate_path, bbox_inches='tight')
plt.show()
print(f'Saved {aggregate_path}')

timeline_df.to_csv(TIMELINE_CSV, index=False)
aggregate_df.to_csv(AGGREGATE_CSV, index=False)
fit_df.to_csv(FIT_CSV, index=False)

summary_df = (
    fit_df.groupby(['KC', 'method'], as_index=False)
    .agg(
        mean_rmse=('rmse', 'mean'),
        mean_r2=('r2', 'mean'),
        student_count=('SubjectID', 'nunique'),
        avg_problem_count=('n_points', 'mean'),
        median_problem_count=('n_points', 'median'),
        avg_gap_rate=('gap_rate', 'mean'),
    )
)

comparison_df = summary_df.pivot(index='KC', columns='method')
comparison_df.columns = [f'{metric}_{method}' for metric, method in comparison_df.columns]
comparison_df = comparison_df.reset_index()
comparison_df['rmse_improvement_v3'] = comparison_df['mean_rmse_baseline'] - comparison_df['mean_rmse_v3']
comparison_df['r2_improvement_v3'] = comparison_df['mean_r2_v3'] - comparison_df['mean_r2_baseline']
comparison_df['better_method_by_rmse'] = np.where(comparison_df['mean_rmse_v3'] < comparison_df['mean_rmse_baseline'], 'v3', 'baseline')
comparison_df['better_method_by_r2'] = np.where(comparison_df['mean_r2_v3'] > comparison_df['mean_r2_baseline'], 'v3', 'baseline')
comparison_order = comparison_df.sort_values('rmse_improvement_v3', ascending=True)['KC'].tolist()

fig, axes = plt.subplots(1, 2, figsize=(13.5, max(5.2, 0.48 * len(comparison_order) + 1.5)), sharey=True)
y = np.arange(len(comparison_order))
comp = comparison_df.set_index('KC').loc[comparison_order]
axes[0].barh(y, comp['rmse_improvement_v3'], color=np.where(comp['rmse_improvement_v3'] >= 0, '#2a7f62', '#b75d3a'))
axes[0].axvline(0, color='#333333', lw=1)
axes[0].set_yticks(y)
axes[0].set_yticklabels(comparison_order)
axes[0].set_title('Fit error change')
axes[0].set_xlabel('Baseline RMSE - V3 RMSE')
axes[0].grid(axis='x', alpha=0.22)
axes[1].barh(y, comp['r2_improvement_v3'], color=np.where(comp['r2_improvement_v3'] >= 0, '#2a7f62', '#b75d3a'))
axes[1].axvline(0, color='#333333', lw=1)
axes[1].set_title('Fit clarity change')
axes[1].set_xlabel('V3 R2 - baseline R2')
axes[1].grid(axis='x', alpha=0.22)
fig.suptitle('Does the LLM gap signal produce cleaner KC learning curves than score alone?', y=1.02, fontsize=14)
fig.tight_layout(rect=(0, 0, 1, 0.94))
fit_comparison_path = FIG_DIR / 'exp18_fit_quality_comparison.png'
fig.savefig(fit_comparison_path, bbox_inches='tight')
plt.show()
print(f'Saved {fit_comparison_path}')

summary_payload = {
    'students': {
        'focus_students': FOCUS_STUDENTS,
        'annotation_students': sorted(annotation_lookup.keys()),
        'n_students_with_timelines': int(timeline_df.SubjectID.nunique()),
        'n_student_kc_timelines': int(timeline_df[['SubjectID', 'KC']].drop_duplicates().shape[0]),
    },
    'timelines': {
        'rows': int(len(timeline_df)),
        'kcs': int(timeline_df['KC'].nunique()),
        'min_kc_problems': MIN_KC_PROBLEMS,
        'min_problem_coverage': MIN_PROBLEMS_PER_KC,
    },
    'kc_coverage': {
        'fit_kcs': fit_kc_columns,
        'excluded_kcs': excluded_kc_columns,
        'counts': kc_problem_counts_local,
    },
    'paths': {
        'timeline_csv': str(TIMELINE_CSV),
        'aggregate_csv': str(AGGREGATE_CSV),
        'fit_csv': str(FIT_CSV),
        'summary_json': str(SUMMARY_JSON),
        'figure_dir': str(FIG_DIR),
    },
    'visualizations': {
        'gap_trajectory_overview': str(FIG_DIR / 'exp18_gap_trajectory_overview.png'),
        'student_gap_signal_matrix': str(FIG_DIR / 'exp18_student_gap_signal_matrix.png'),
        'validated_student_gap_timelines': str(FIG_DIR / 'exp18_validated_student_gap_timelines.png'),
        'aggregate_gap_vs_score_curves': str(aggregate_path),
        'fit_quality_comparison': str(fit_comparison_path),
    },
    'per_timeline_fit_summary': summary_df.sort_values(['KC', 'method']).to_dict(orient='records'),
    'aggregate_curve_fit_summary': aggregate_fit_df.sort_values(['KC', 'method']).to_dict(orient='records'),
    'excluded_kc_summary': excluded_kc_summary_df.to_dict(orient='records'),
    'comparison_by_kc': comparison_df.sort_values('KC').to_dict(orient='records'),
}

with open(SUMMARY_JSON, 'w') as f:
    json.dump(summary_payload, f, indent=2, default=lambda obj: obj.item() if hasattr(obj, 'item') else str(obj))

print(f'Saved {TIMELINE_CSV}')
print(f'Saved {AGGREGATE_CSV}')
print(f'Saved {FIT_CSV}')
print(f'Saved {SUMMARY_JSON}')
display(summary_df.sort_values(['KC', 'method']))
display(comparison_df[['KC', 'mean_rmse_baseline', 'mean_rmse_v3', 'rmse_improvement_v3', 'mean_r2_baseline', 'mean_r2_v3', 'r2_improvement_v3', 'better_method_by_rmse', 'better_method_by_r2']])
if not excluded_kc_summary_df.empty:
    display(excluded_kc_summary_df)


## Reading the Results

Use the example student plots to see whether the KC-specific scores flatten out or improve after earlier misses. In the aggregate grid, lower error curves and a tighter power-law fit indicate a cleaner learning trajectory.

The main comparison is in the summary table: if V3 tags are genuinely identifying persistent gaps, they should produce lower RMSE and higher $R^2$ for the affected KCs than the problem-level baseline.

If you want to extend this notebook later, the CSV exports already contain the full timeline-level data and the aggregate curves used for the figures.